# rhino-poc — Colab

**Baslamadan:** Runtime > Change runtime type > **GPU (T4)**.

Colab burada sadece **GPU gereken adim** icin var: COLMAP `patch_match_stereo` (MVS).
Geri kalan her sey (kare secimi, SfM, olcek, FLAME, olcum) yerelde de kosar — bkz. `PLAN.md` SS4.

Hucreleri sirayla kos.

In [ ]:
!nvidia-smi

## 1. Repoyu cek
Kod GitHub'dan gelir; Drive'a repo yuklemeye gerek yok.

In [ ]:
import os
REPO = '/content/rhino-poc'
if os.path.isdir(REPO + '/.git'):
    !cd {REPO} && git fetch --quiet origin && git reset --hard origin/main
else:
    !git clone --quiet https://github.com/Daml4Yilmaz/rhino-poc.git {REPO}
!cd {REPO} && git log --oneline -1

## 2. Drive'i bagla (veri icin)
Kod repodan, **veri Drive'dan** gelir. Videonu/yakalamani Drive'a koy:
`MyDrive/rhino-poc-data/` icine.

Ciktilar da Drive'a yazilir — oturum dusse de kaybolmaz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/rhino-poc-data'
!mkdir -p {DATA}
!ls -la {DATA}

## 3. COLMAP (CUDA'li)
Once hizli yol: conda-forge GPU derlemesi. Dogrulama hucresi CUDA gosterirse alttaki
kaynaktan derleme hucresini ATLA.

In [ ]:
%%bash
cd /opt
if [ ! -x /opt/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
if [ ! -x /opt/colmapenv/bin/colmap ]; then
  /opt/bin/micromamba create -y -q -p /opt/colmapenv -c conda-forge 'colmap=*=gpu*' \
    || echo 'GPU derlemesi bulunamadi - kaynaktan derleme gerekebilir'
fi

In [ ]:
import os
for c in ('/opt/colmapenv/bin/colmap', '/usr/local/colmap/bin/colmap'):
    if os.path.exists(c):
        os.system('ln -sf ' + c + ' /usr/local/bin/colmap')
        break
!colmap -h 2>&1 | head -3

**PATH uyarisi:** conda klasorunu PATH'in basina EKLEME — icindeki python sistem
python'unu golgeler ve `cv2` kirilir. Sadece symlink kullan (ustteki hucre bunu yapiyor).

## 4. Python bagimliliklari + repo kurulumu

In [ ]:
!pip install -q opencv-contrib-python open3d trimesh pycolmap typer pandas pillow
!pip install -q {REPO}
import poc; print('poc hazir')

**`-e` (editable) KULLANMA** — Jupyter ayni oturumda import edemez, restart ister.
Repoda kod degisirse: 1. hucreyi (git reset) ve bu hucreyi tekrar kos.

## 5A. TEST — duz .mov/.mp4 video ile

ARKit verisi olmadan sadece **fotogrametri hattini** dogrular:
kare secimi -> SfM -> MVS -> mesh.

**Cikan model BIRIMSIZ olur** (`model_unitless.glb`). Aci ve Goode orani anlamli;
mm cinsinden uzunluk/genislik/sapma URETILEMEZ — onun icin Stray Scanner kaydi gerekir (5B).

Videoyu Drive'a koy: `MyDrive/rhino-poc-data/test.mov`

In [ ]:
VIDEO = DATA + '/test.mov'
OUT   = DATA + '/test_out'
!poc process {VIDEO} --out {OUT} --n-frames 300 --max-dim 1600

## 5B. GERCEK — Stray Scanner yakalamasi

Klasoru oldugu gibi Drive'a kopyala: `MyDrive/rhino-poc-data/vaka_001_stray/`
icinde `rgb.mp4`, `odometry.csv`, `camera_matrix.csv`, `depth/`, `confidence/` olmali.

Bu yolda olcek de kosar: `scale.json` -> `agreement_pct` < 1.5 ve `scale_verified: true` bekliyoruz.

In [ ]:
CAPTURE = DATA + '/vaka_001_stray'
OUT     = DATA + '/vaka_001'
!poc process {CAPTURE} --out {OUT}

## 6. Sonuclar

`OUT` icinde: `frames/`, `frames_index.json`, `colmap/`, `mesh_raw.ply`,
`scale.json` (sadece 5B), `model.glb` / `model_unitless.glb`.

GLB'yi [gltf.report](https://gltf.report) veya Blender'da ac.
5B'de kafa bbox ~200-250 mm cikmali.

In [ ]:
import json, os
print(sorted(os.listdir(OUT)))
sj = os.path.join(OUT, 'scale.json')
if os.path.exists(sj):
    print(json.dumps(json.load(open(sj)), indent=2))